Scarping for https://www.allfreecrochet.com/

In [1]:
import requests
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
import os

headers = {"User-Agent": "Mozilla/5.0"}

target_dir = "../frontend/src/assets/images"
os.makedirs(target_dir, exist_ok=True)

In [2]:
def scrape_product(product_link):
    row = {}
    os.makedirs(target_dir, exist_ok=True)

    try:
        r = requests.get(product_link, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Connection error for {product_link}: {e}")
        return row

    # title
    title = soup.find("h1", class_="articleHeadline")
    if title:
        row["title"] = title.get_text(strip=True)
    else:
        row["title"] = ""  

    # description
    desc = soup.select_one("div.articleAttrSection")
    description = desc.get_text(strip=True)
    row["description"] = description

    sections = soup.select("div.articleAttrSection")

    # find details in the sections
    for section in sections:
        label_span = section.find("span", class_="attrLabel")
        img = section.find("img")

        if label_span:
            key = label_span.get_text(strip=True)
            
            # check if next span exist only in this section
            value_span = label_span.find_next_siblings("span")

            if value_span:
                value = value_span[0].get_text(strip=True)
            else:
                # No span sibling, get text in the parent <p> minus the label
                full_text = label_span.parent.get_text(" ", strip=True)
                # remove the label text from the beginning
                value = full_text.replace(key, "", 1).strip()
            
            row[key] = value

        # skill level is in img tag, so check it separately
        elif img:
            skill_level = img.get("alt", "").strip()
            if skill_level ==  "Easy":
                skill_level = "Beginner"
            row["skill_level"] = skill_level         

    # image extraction and download
    img_container = soup.find("div", class_="imgContainer")
    img_tag = img_container.find("img") if img_container else None

    img_url = None
    if img_tag:
        img_url = "https:" + img_tag.get("src")
    row["img_url"] = img_url

    if img_url:
        safe_filename = re.sub(r'[^\w\s-]', '', row["title"]).strip().replace(' ', '_')
        save_path = os.path.join(target_dir, f"{safe_filename}.jpg")

        try:
            img_res = requests.get(img_url, headers=headers, stream=True, timeout=10)
            if img_res.status_code == 200 and 'image' in img_res.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    for chunk in img_res.iter_content(1024):
                        f.write(chunk)
                row["local_path"] = f"{safe_filename}.jpg"
            else:
                print(f"Invalid image response for: {row['title']}")
        except Exception as e:
            print(f"Error downloading image for {row['title']}: {e}")

    return row


In [3]:
test = "https://www.allfreecrochet.com/Crochet-Afghan-Patterns/Spring-Breeze-Baby-Blanket-187303"
scrape_product(test)

{'title': 'Spring Breeze Baby Blanket',
 'description': '"Wrap your little one in the soft, airy charm of the Spring Breeze Baby Blanket – Free Crochet Pattern that’s as refreshing as a warm spring morning! This delightful design features a soothing rhythm of two rows of cozy double crochet, followed by a playful eyelet row that adds just the right touch of texture. Then, just when you think you’ve got the pattern down, a second, slightly different eyelet row keeps things interesting and fun for your hook. The result is a light and breezy baby blanket with a beautiful balance of simplicity and whimsy which is perfect for springtime snuggles or thoughtful handmade gifts!"',
 'skill_level': 'Intermediate',
 'Crochet Hook': 'H/8 or 5 mm hook',
 'Yarn Weight': '(4) Medium Weight/Worsted Weight and Aran (16-20 stitches to 4 inches)',
 'Finished Size': '26.45 x 25.60 inches approx',
 'img_url': 'https://irepo.primecp.com/2025/06/601674/1749590604_117446_Large600_ID-5853120.png?v=5853120',
 '

In [4]:
base_url = "https://www.allfreecrochet.com"
rows = []

response = requests.get(base_url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")

cards = soup.select("div.articleCell")

for card in cards:
    link = card.select_one("a[href]")
    product_link = base_url + link["href"] 

    # img_tag = card.select_one("img")
    # image_link = img_tag["src"] if img_tag else None

    # name_tag = card.select_one("figcaption a")
    # name = name_tag.get_text(strip=True) if name_tag else None

    product_data = scrape_product(product_link)

    # product_data["name"] = name
    product_data["pattern_link"] = product_link
    # product_data["image_link"] = image_link

    rows.append(product_data)
    time.sleep(1)


print("Scraping completed. Total products scraped:", len(rows))
df = pd.DataFrame(rows)

Scraping completed. Total products scraped: 41


In [9]:
df.head()

,title,description,skill_level,Crochet Hook,Yarn Weight,Finished Size,img_url,local_path,pattern_link,Crochet Gauge,Notes
0,Quick n' Cozy Crochet Afghan,The Quick n' Cozy Crochet Afghan is the ultima...,Beginner,P/16 or 11.5 mm hook,(6) Super Bulky/Super Chunky (4-11 stitches fo...,"44 x 55""",https://irepo.primecp.com/2015/06/225420/Quick...,Quick_n_Cozy_Crochet_Afghan.jpg,https://www.allfreecrochet.com/Crochet-Afghan-...,NaN,NaN
2,World's Easiest Afghan,If you've been searching for easy afghan patte...,Beginner,H/8 or 5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,Measurement: 48 x 60”,https://irepo.primecp.com/2014/12/204013/World...,Worlds_Easiest_Afghan.jpg,https://www.allfreecrochet.com/Crochet-Afghan-...,"13 sc and 14 rows = 4"" (10cm)",NaN
3,Ribbed Fingerless Gloves,"""These crochet ribbed fingerless gloves are ma...",Beginner,4.5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,NaN,https://irepo.primecp.com/2026/03/609601/17728...,Ribbed_Fingerless_Gloves.jpg,https://www.allfreecrochet.com/Gloves-and-Mitt...,NaN,NaN
4,Chunky Adult Slippers,Learn how to crochet an adorable pair of Chunk...,Beginner,M/13 or 9 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,NaN,https://irepo.primecp.com/2020/11/471382/Chunk...,Chunky_Adult_Slippers.jpg,https://www.allfreecrochet.com/Socks-and-Slipp...,Gauge 2 sc = 1 inch,NaN
5,Quick Little Bag,"""This Quick Little Bag is just that, and it ca...",Beginner,I/9 or 5.5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,5 inch square,https://irepo.primecp.com/2015/09/238794/Quick...,Quick_Little_Bag.jpg,https://www.allfreecrochet.com/Crochet-Bag-Pat...,NaN,NaN


In [10]:
df = df.loc[~df["img_url"].isna(),]

In [11]:
df.head()

,title,description,skill_level,Crochet Hook,Yarn Weight,Finished Size,img_url,local_path,pattern_link,Crochet Gauge,Notes
0,Quick n' Cozy Crochet Afghan,The Quick n' Cozy Crochet Afghan is the ultima...,Beginner,P/16 or 11.5 mm hook,(6) Super Bulky/Super Chunky (4-11 stitches fo...,"44 x 55""",https://irepo.primecp.com/2015/06/225420/Quick...,Quick_n_Cozy_Crochet_Afghan.jpg,https://www.allfreecrochet.com/Crochet-Afghan-...,NaN,NaN
2,World's Easiest Afghan,If you've been searching for easy afghan patte...,Beginner,H/8 or 5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,Measurement: 48 x 60”,https://irepo.primecp.com/2014/12/204013/World...,Worlds_Easiest_Afghan.jpg,https://www.allfreecrochet.com/Crochet-Afghan-...,"13 sc and 14 rows = 4"" (10cm)",NaN
3,Ribbed Fingerless Gloves,"""These crochet ribbed fingerless gloves are ma...",Beginner,4.5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,NaN,https://irepo.primecp.com/2026/03/609601/17728...,Ribbed_Fingerless_Gloves.jpg,https://www.allfreecrochet.com/Gloves-and-Mitt...,NaN,NaN
4,Chunky Adult Slippers,Learn how to crochet an adorable pair of Chunk...,Beginner,M/13 or 9 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,NaN,https://irepo.primecp.com/2020/11/471382/Chunk...,Chunky_Adult_Slippers.jpg,https://www.allfreecrochet.com/Socks-and-Slipp...,Gauge 2 sc = 1 inch,NaN
5,Quick Little Bag,"""This Quick Little Bag is just that, and it ca...",Beginner,I/9 or 5.5 mm hook,(4) Medium Weight/Worsted Weight and Aran (16-...,5 inch square,https://irepo.primecp.com/2015/09/238794/Quick...,Quick_Little_Bag.jpg,https://www.allfreecrochet.com/Crochet-Bag-Pat...,NaN,NaN


In [12]:
df.to_csv("../data/all_free_crochet_patterns.csv", index=False)